# Khowar OCR Demo

Scanned Khowar (Nastaliq) text to digital text, using a DeepSeek-OCR-2 fine-tune (`zahidazam714/khowar-ocr-v0.1`).

Requirements: Kaggle GPU (T4), Internet on, a Hugging Face token.

In [ ]:
# torchao is uninstalled because the preinstalled version conflicts with peft
!pip uninstall -y -q torchao
!pip install -q gradio einops addict easydict

In [ ]:
import re
import sys
import time
from io import StringIO

import gradio as gr
import torch
from huggingface_hub import login
from transformers import AutoModel, AutoTokenizer

login("YOUR_HF_TOKEN")

OCR_REPO = "zahidazam714/khowar-ocr-v0.1"

ocr_tok = AutoTokenizer.from_pretrained(OCR_REPO, trust_remote_code=True)
ocr_model = AutoModel.from_pretrained(
    OCR_REPO,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
).eval().cuda()

print("Model loaded")

In [ ]:
def clean_output(raw):
    skip = ("=====", "BASE:", "PATCHES:")
    lines = [l for l in raw.splitlines() if l.strip() and not l.startswith(skip)]
    return "\n".join(lines).strip()


def ocr_ui(img):
    if img is None:
        return "Upload an image first.", ""

    path = "/tmp/ui_input.png"
    img.convert("RGB").save(path)

    # infer() prints its result to stdout, so capture it
    old = sys.stdout
    sys.stdout = buf = StringIO()
    t0 = time.time()
    try:
        ocr_model.infer(
            ocr_tok,
            prompt="<image>\nFree OCR.",
            image_file=path,
            output_path="/tmp/ocr_out",
            base_size=1024,
            image_size=768,
            crop_mode=True,
            save_results=False,
        )
    finally:
        sys.stdout = old
    elapsed = time.time() - t0

    raw = buf.getvalue()
    text = clean_output(raw)

    base = re.search(r"BASE:\s+torch\.Size\(\[(.*?)\]\)", raw)
    patches = re.search(r"PATCHES:\s+torch\.Size\(\[(.*?)\]\)", raw)
    details = (
        "Model: DeepSeek-OCR-2 + Khowar LoRA (khowar-ocr-v0.1)\n"
        f"Global view tokens: [{base.group(1) if base else '-'}]\n"
        f"Local crop tokens: [{patches.group(1) if patches else '-'}]\n"
        f"Inference time: {elapsed:.1f}s on {torch.cuda.get_device_name(0)}"
    )
    return text, details

In [ ]:
with gr.Blocks(title="Khowar OCR - NeuraFlix") as demo:
    gr.Markdown("# Khowar OCR\nScanned Khowar (Nastaliq) text to digital text")
    with gr.Row():
        inp = gr.Image(type="pil", label="Input: Khowar image")
        with gr.Column():
            out = gr.Textbox(label="Output: recognized Khowar text", lines=10, rtl=True)
            details = gr.Textbox(label="Processing details", lines=4)
    btn = gr.Button("Run OCR", variant="primary")
    btn.click(ocr_ui, inp, [out, details])

# open the printed public link in a new browser tab for screenshots
demo.launch(share=True)